# Local-tree inference

{class}`~ancestree.local_tree_inference.LocalTreeInference` infers the ancestral allele from genotypes alone: no outgroups and no input ARG. It infers a dated local tree per genomic window (a PSMC$^\prime$-style pairwise-coalescent HMM followed by per-window UPGMA) and then runs the same likelihood kernel as {class}`~ancestree.inference.ARGBasedInference` over the inferred genealogy.

## Loading the tree sequence

The tree sequence supplies the truth and the true-ARG ceiling only. The panel is all 8 haplotypes (6 ingroup + 2 outgroup), and every tip enters the kernel symmetrically, as in ARG mode. The ingroup and outgroup names only place the reporting node at the ingroup MRCA.


In [1]:
import tskit

ts = tskit.load("quickstart.trees")  # truth + the true-ARG ceiling only
ingroup = [f"i{i}" for i in range(6)]
outgroup = ["o0", "o1"]
print(f"{ts.num_samples} haplotypes, {ts.num_sites} sites")


8 haplotypes, 224 sites


The inference reads a VCF exported from it, so the local trees are built from the genotypes alone and the true genealogy is never read.


In [2]:
with open("quickstart.vcf", "w") as vcf:
    ts.write_vcf(vcf, contig_id="1", individual_names=ingroup + outgroup)


## Running the inference

Local-tree mode is built from genotypes and used like the other modes, with {paramref}`window <ancestree.local_tree_inference.LocalTreeInference.window>` a SNP-scaled width. The result is graded on the sites polymorphic within the ingroup, as in the {doc}`quickstart`, and compared with the true-ARG ceiling, the true genealogy run through {meth}`Inference.from_arg() <ancestree.inference.Inference.from_arg>`, which separates the cost of inferring trees from that of inferring the allele on them.

For whole-genome panels, {paramref}`chunk_size <ancestree.local_tree_inference.LocalTreeInference.chunk_size>` (and optionally {paramref}`n_workers <ancestree.local_tree_inference.LocalTreeInference.n_workers>`) streams the genome in independent chunks, so the pairwise HMM's $\binom{n}{2}$ TMRCA matrix, with $n$ the number of haplotypes, is never held for the whole sequence at once. See {class}`~ancestree.local_tree_inference.LocalTreeInference` for the chunking, halo, rate-map and accessibility-mask options.

Each window's posterior is marginalised over an ensemble of genealogies drawn from the pairwise HMM's own posterior, rather than resting on a single agglomerated tree. {paramref}`n_ensemble <ancestree.local_tree_inference.LocalTreeInference.params.n_ensemble>` sets the ensemble size, with cost linear in it, and `None` selects the plug-in estimate. Peak memory is set by {paramref}`member_chunk <ancestree.local_tree_inference.LocalTreeInference.params.member_chunk>`, not by the ensemble size, so a larger ensemble costs time rather than memory.

In [3]:
import ancestree as anc

lt = anc.Inference.from_local_tree(
    "quickstart.vcf", anc.JC69(), mu=5e-8, rec_rate=1e-8,
    ingroup_samples=ingroup, outgroup_samples=outgroup,
    sequence_length=ts.sequence_length, n_ensemble=64,
)
res_lt = list(lt.infer())


INFO:ancestree.CyVCF2Source: Genotypes read as ploidy 1; pass ploidy= to override
INFO:ancestree.CyVCF2Source: Reading variants from quickstart.vcf (8 haplotype samples)
INFO:ancestree.LocalTreeInference: Inferring local trees from genotypes (8 samples, window=8snp)
INFO:ancestree.LocalTreeInference: Using 6 ingroup sample(s): i0, i1, i2, i3, i4, i5
INFO:ancestree.LocalTreeInference: Using 2 outgroup sample(s): o0, o1
LocalTreeInference: 100%|██████████| 1/1 [00:24<00:00, 24.33s/ segments]
INFO:ancestree.LocalTreeInference: Focal node: reported at the ingroup_mrca


The result is graded on the sites polymorphic within the ingroup against the true allele at the ingroup MRCA.


In [4]:
keep = anc.PolymorphicSiteFilter(samples=ingroup)
truth = anc.Grade.truth_at_focal(ts, ingroup_samples=ingroup, outgroup_samples=outgroup)
anc.Grade(res_lt, truth, filter=keep)


118 sites: MAP recovery 98.3%; mean Brier = 0.028

The true genealogy run through ARG mode gives the ceiling the inferred trees are measured against.


In [5]:
ceiling = anc.Inference.from_arg(
    ts, anc.JC69(), mu=5e-8,
    ingroup_samples=ingroup, outgroup_samples=outgroup,
)
res_arg = list(ceiling.infer())


INFO:ancestree.ARGBasedInference: Inferring the ancestral allele at 224 sites over 29 local trees from the ARG
INFO:ancestree.ARGBasedInference: Using 6 ingroup sample(s): i0, i1, i2, i3, i4, i5
INFO:ancestree.ARGBasedInference: Using 2 outgroup sample(s): o0, o1
ARGBasedInference: 100%|██████████| 29/29 [00:00<00:00, 390.10 trees/s]
INFO:ancestree.ARGBasedInference: Focal node: reported at the ingroup_mrca; 4 tree(s) where the ingroup is not monophyletic, so its MRCA subtends outgroup tips


Side by side, the inferred trees and the true genealogy score as follows.


In [6]:
print(f"{'mode':<28} {'MAP recovery':>13} {'mean Brier':>12}")
for label, r in (("Local-tree mode", res_lt), ("ARG mode, true ARG (ceiling)", res_arg)):
    g = anc.Grade(r, truth, filter=keep)
    print(f"{label:<28} {g.map_recovery:>13.1%} {g.brier:>12.3f}")


mode                          MAP recovery   mean Brier
Local-tree mode                      98.3%        0.028
ARG mode, true ARG (ceiling)         98.3%        0.031


In [7]:
g_lt, g_arg = anc.Grade(res_lt, truth, filter=keep), anc.Grade(res_arg, truth, filter=keep)
assert g_lt.map_recovery >= g_arg.map_recovery - 0.01
assert g_lt.brier <= g_arg.brier + 0.01


:::{note}
On this small panel the inferred trees reach the true-ARG ceiling. On larger panels a gap opens, and it widens with a mis-sized window, mis-set rates or phasing switch error.

{paramref}`mu <ancestree.local_tree_inference.LocalTreeInference.mu>` and {paramref}`rec_rate <ancestree.local_tree_inference.LocalTreeInference.rec_rate>` need only be approximately correct: the time grid is recalibrated from the data, and the SNP-scaled {paramref}`window <ancestree.local_tree_inference.LocalTreeInference.window>` absorbs recombination-rate misspecification.
:::


## Inspect the inferred genealogy

{meth}`LocalTreeInference.point_tree_sequence() <ancestree.local_tree_inference.LocalTreeInference.point_tree_sequence>` returns the plug-in genealogy as a {class}`tskit.TreeSequence`: one agglomerated tree per window, branch lengths in generations, with the observed genotypes included as sites, so the trees the assignments rest on can be inspected or written out.


In [8]:
pt = lt.point_tree_sequence()
print(f"plug-in: {pt.num_trees} local trees across {pt.num_sites} sites")


plug-in: 28 local trees across 224 sites


{meth}`LocalTreeInference.tree_sequences() <ancestree.local_tree_inference.LocalTreeInference.tree_sequences>` instead yields what the posterior was marginalised over, one stretch of genome at a time: each item is an `(interval, members)` pair, so peak memory holds one stretch. In ensemble mode `members` holds the `n_ensemble` draws, and with `n_ensemble=None` the plug-in tree alone. The draws carry topology and times but no sites, and are a pure function of {paramref}`ensemble_seed <ancestree.local_tree_inference.LocalTreeInference.params.ensemble_seed>`.


In [9]:
(lo, hi), members = next(iter(lt.tree_sequences()))
first = next(members)
print(f"stretch [{lo:,.0f}, {hi:,.0f}): first member has "
      f"{first.num_trees} trees, {first.num_sites} sites")
print(f"ensemble size: {lt.n_ensemble}")


stretch [0, 50,000): first member has 28 trees, 0 sites
ensemble size: 64


## Saving the output

{meth}`LocalTreeInference.to_vcf() <ancestree.local_tree_inference.LocalTreeInference.to_vcf>` writes the annotations as a VCF, annotating the records of the input VCF (see {doc}`io`).


In [10]:
lt.to_vcf("annotated.vcf.gz", posteriors=res_lt);


INFO:ancestree.VCFWriter: Wrote 224 annotated sites to annotated.vcf.gz


In [11]:
anc.Reader("annotated.vcf.gz").head(3)


chrom   pos  alleles  AA       A       C       G       T
1       556  G/C      G   0.0000  0.0003  0.9997  0.0000
1       927  G/T      T   0.0000  0.0000  0.0000  1.0000
1      1031  T/A      T   0.0003  0.0000  0.0000  0.9997

The inferred genealogies can be saved as well. {meth}`LocalTreeInference.to_arg() <ancestree.local_tree_inference.LocalTreeInference.to_arg>` writes the point-estimate genealogy, with the annotations on its sites ({meth}`LocalTreeInference.point_tree_sequence() <ancestree.local_tree_inference.LocalTreeInference.point_tree_sequence>`).

In [12]:
lt.to_arg("point.trees", posteriors=res_lt);

INFO:ancestree.TskitWriter: Wrote 224 annotated sites to point.trees


{meth}`LocalTreeInference.tree_sequences() <ancestree.local_tree_inference.LocalTreeInference.tree_sequences>` yields the ensemble draws for each genome chunk of {paramref}`chunk_size <ancestree.local_tree_inference.LocalTreeInference.chunk_size>`, which is inferred independently, so a draw does not continue into the next chunk. Unlike the point estimate, the draws carry no sites and are written unannotated.

In [13]:
for (lo, hi), draws in lt.tree_sequences():
    for k, draw in enumerate(draws):
        draw.dump(f"draw_{lo:.0f}_{k}.trees")
    print(f"stretch [{lo:,.0f}, {hi:,.0f}): wrote {k + 1} draws")

stretch [0, 50,000): wrote 64 draws


In [14]:
from pathlib import Path

Path("annotated.vcf.gz").unlink(missing_ok=True)
Path("point.trees").unlink(missing_ok=True)
for draw in Path().glob("draw_*.trees"):
    draw.unlink()
Path("quickstart.vcf").unlink(missing_ok=True)
